# Model Development — Used Heavy Equipment Pricing

**Business task:** Predict historical auction `SalePrice` from five features available at the time of a sale. This notebook reproduces the final **development benchmark** configurations from the original analysis. It does **not** claim independent test performance.

**Models:** 50-tree Random Forest with train-only preprocessing; 500-iteration CatBoost with missing numerical values preserved.

**Evaluation:** Three five-year training windows, each followed by a two-year chronological validation window. Running all experiments on 401,125 rows may take substantial compute time.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate package modules from the repository root or notebooks/ directory.
project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src').is_dir():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks folder.')
sys.path.insert(0, str(project_root))
from src.paths import raw_data_path
from src.preprocessing import clean_transactions, catboost_features, CATEGORICAL, PREDICTORS
from src.modeling import WINDOWS, run_walkforward, rf_pipeline, catboost_model, error_metrics
from src.evaluate import comparison_table, by_period_table, missing_age_error_report

## Dataset and cleaning rules
The project applies identical deterministic quality rules to training and validation records. No future target statistics are used to fit preprocessing.

In [2]:
# The raw CSV is intentionally excluded from version control.
raw_df = pd.read_csv(raw_data_path(), parse_dates=['saledate'], low_memory=False)
cleaned = clean_transactions(raw_df)
print(f'Loaded {len(cleaned):,} transactions from {raw_df.saledate.min():%Y-%m-%d} to {raw_df.saledate.max():%Y-%m-%d}')
print('Invalid or unknown equipment ages:', int(cleaned.EquipmentAge.isna().sum()))

Loaded 401,125 transactions from 1989-01-17 to 2011-12-30
Invalid or unknown equipment ages: 38198


## Chronological development experiments
Model selection and feature decisions were made with repeated exposure to these periods. Scores therefore measure **development-validation performance**.

In [3]:
display(pd.DataFrame(WINDOWS))

,Experiment,TrainStart,TrainEnd,ValidStart,ValidEnd
0,A,2001,2005,2006,2007
1,B,2003,2007,2008,2009
2,C,2005,2009,2010,2011


## Models and preprocessing
`ModelID` is a categorical identifier, not a continuous numeric measurement. The Random Forest uses median imputation with missing-value indicators for numerical inputs and one-hot encoding for categorical inputs. CatBoost accepts missing numeric inputs and uses an explicit categorical missing label.

In [4]:
print('Random Forest:')
print(rf_pipeline())
print('\nCatBoost:')
print(catboost_model().get_params())

Random Forest:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median'),
                                                  ['EquipmentAge',
                                                   'MachineHours_Clean']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['ProductGroup', 'ModelID',
                                                   'ProductSize'])])),
                ('model',
                 RandomForestRegressor(max_depth=24, max_features=0.5,
                                       min_samples_leaf=5, n_estimators=50,
                                       n_jobs=-1, random_state=42))])

CatBoost:
{'iterations': 500, '

## Reproduce walk-forward model comparisons
This step fits **six models** across the three historical windows. All metrics are calculated directly from predictions. No model evaluation numbers are manually inserted.

In [5]:
walkforward, validation_predictions = run_walkforward(cleaned, include_predictions=True)
display(walkforward.round(2))
display(comparison_table(walkforward).round(2))

,Experiment,Model,TrainingPeriod,ValidationPeriod,TrainingTransactions,ValidationTransactions,MAE,MedianAE,P90AE,MeanSignedError,UnderestimationRatePercent,ErrorAbove10000Percent,MAPEPercent
0,A,Random Forest,2001-2005,2006-2007,90436,53893,7316.82,4259.02,17780.18,3859.64,63.35,22.98,22.31
1,A,CatBoost,2001-2005,2006-2007,90436,53893,7386.08,4122.22,18478.59,2829.44,57.78,22.79,22.42
2,B,Random Forest,2003-2007,2008-2009,109489,83616,7879.70,5156.92,18477.76,-3086.29,32.23,26.66,30.72
3,B,CatBoost,2003-2007,2008-2009,109489,83616,8676.73,5734.27,20664.43,-4257.39,28.18,31.06,33.63
4,C,Random Forest,2005-2009,2010-2011,157972,68587,7464.70,4283.12,17986.47,2919.30,57.47,23.41,24.23
5,C,CatBoost,2005-2009,2010-2011,157972,68587,7200.14,4189.99,17422.94,1207.09,49.44,22.08,23.87


,Model,WalkForwardAverageMAE,WalkForwardMAEStd,Periods
1,Random Forest,7553.74,291.81,3
0,CatBoost,7754.32,804.23,3


## Save reproducible evaluation artifacts
Prediction CSVs include no personally identifying fields. Both model metrics and 2010–2011 transaction-level results are generated from new fits.

In [6]:
output_dir = project_root / 'reports' / 'tables'
output_dir.mkdir(parents=True, exist_ok=True)
walkforward.to_csv(output_dir / 'walkforward_recomputed.csv', index=False)
comparison_table(walkforward).to_csv(output_dir / 'model_comparison_recomputed.csv', index=False)
validation_predictions[('C','CatBoost')].to_csv(output_dir / 'catboost_2010_2011_predictions.csv', index=False)
print('Saved recomputed development-validation results to:', output_dir)

Saved recomputed development-validation results to: C:\Users\A\Desktop\sample_project_1\used-heavy-equipment-price-prediction-github\reports\tables


## Interpretation and limitations

The original experimental notebook reported a three-period mean MAE of approximately **$7,776.90 (Random Forest)** and **$7,754.32 (CatBoost)**. These are *historical reference values only* until the commands above reproduce them in your environment. CatBoost did not win in every period.

The selected feature set, model configurations, and validation periods were investigated repeatedly. A defensible claim about generalization requires **new, truly unseen future observations** or an explicitly pre-registered evaluation. The previously inspected 2012 dataset does not qualify.